# PyRIT Scan

`pyrit_scan` is the primary command-line tool for running automated security assessments and red teaming attacks against AI systems. It leverages [scenarios](../code/scenarios/0_scenarios.ipynb) to define attack techniques and supports flexible [configuration](../code/setup/1_configuration.ipynb) for targeting different AI endpoints.

For configuration setup, see [Configuration](../getting_started/configuration.md).

For scenario-specific examples, see [AIRT](airt.ipynb), [Foundry](foundry.ipynb), and [Garak](garak.ipynb).

Note in this doc the ! prefaces all commands in the terminal so we can run in a Jupyter Notebook.

## Starting a Backend Server

`pyrit_scan` is a thin client that talks to a PyRIT backend server (by default at `http://localhost:8000`).
Before running any command that reaches the backend (listing scenarios, running a scan, etc.) you need a
server. Start a local one with `--start-server`; it launches a detached `pyrit_backend` process that stays
up and is reused by every command below. We stop it again at the end of the notebook.

In [ ]:
!pyrit_scan --start-server

Starting server at http://localhost:8000...
Server ready (PID 45120)
Server is running at http://localhost:8000


## Quick Start

For help:

In [ ]:
!pyrit_scan --help

usage: pyrit_scan [-h] [--server-url SERVER_URL] [--start-server]
                  [--stop-server] [--config-file CONFIG_FILE]
                  [--log-level LOG_LEVEL] [--request-timeout REQUEST_TIMEOUT]
                  [--list-scenarios] [--list-initializers] [--list-targets]
                  [--list-converters] [--list-datasets]
                  [--add-initializer FILE [FILE ...]] [--target TARGET]
                  [--initializers INITIALIZERS [INITIALIZERS ...]]
                  [--techniques SCENARIO_TECHNIQUES [SCENARIO_TECHNIQUES ...]]
                  [--max-concurrency MAX_CONCURRENCY]
                  [--max-retries MAX_RETRIES] [--memory-labels MEMORY_LABELS]
                  [--dataset-names DATASET_NAMES [DATASET_NAMES ...]]
                  [--max-dataset-size MAX_DATASET_SIZE]
                  [scenario_name]

PyRIT Scanner - Run AI security scenarios from the command line.

Requires a running PyRIT backend server. Use --start-server to launch one,
or connect

### Discovery

List all available scenarios:

In [ ]:
!pyrit_scan --list-scenarios


Available Scenarios:

  adaptive.text_adaptive
    Class: TextAdaptive
    Description:
      Adaptive text-attack scenario. Selects techniques per-objective via an
      epsilon-greedy selector over the set of selected techniques.
      ``prompt_sending`` runs as the baseline comparison and is excluded from
      the adaptive technique pool.
    Aggregate Techniques:
      - all, default, single_turn, multi_turn
    Available Techniques (11):
      role_play, many_shot, tap, crescendo_simulated, red_teaming,
      context_compliance, crescendo_movie_director, crescendo_history_lecture,
      crescendo_journalist_interview, pair, violent_durian
    Default Technique: default
    Default Datasets (7):
      airt_hate, airt_fairness, airt_violence, airt_sexual, airt_harassment,
      airt_misinformation, airt_leakage
    Supported Parameters:
      - objective_target (any): Target system under attack: a registered target name or a PromptTarget instance.
      - scenario_techniques (any)

**Tip**: You can also discover user-defined scenarios by providing initialization scripts:

```shell
pyrit_scan --list-scenarios --initialization-scripts ./my_custom_initializer.py
```

This will load your custom scenario definitions and include them in the list.

## Initializers

PyRITInitializers are how you can configure the CLI scanner. PyRIT includes several built-in initializers you can use with the `--initializers` flag.

The `--list-initializers` command shows all available initializers. Initializers are referenced by their filename (e.g., `target`, `scorer`, `simple`) regardless of which subdirectory they're in.

List the available initializers using the --list-initializers flag.

In [ ]:
!pyrit_scan --list-initializers


Available Initializers:

  load_default_datasets
    Class: LoadDefaultDatasets
    Required Environment Variables: None
    Supported Parameters:
      - dataset_names: Explicit dataset names to load. Overrides the scenario-default selection.
      - tags: Load datasets whose metadata matches these tags. Overrides scenario-default selection.
    Description:
      Load datasets into memory so scenarios can run. By default this loads
      the datasets required by all registered scenarios. Pass
      ``dataset_names`` to load specific datasets by name, or ``tags`` to
      select datasets by metadata.

  preload_scenario_metadata
    Class: PreloadScenarioMetadata
    Required Environment Variables: None
    Description:
      Instantiate every registered scenario once to warm the metadata cache.

  scorer
    Class: ScorerInitializer
    Required Environment Variables: None
    Supported Parameters:
      - tags [default: ['default']]: Tags for filtering (e.g., ['default'])
    Descr

### Running Scenarios

You need a single scenario to run, you need two things:

1. A Scenario. Many are defined in `pyrit.scenario.scenarios`. But you can also define your own in initialization_scripts.
2. Initializers (which can be supplied via `--initializers` or `--initialization-scripts` or `initializers` section of config file (see [here](../getting_started/pyrit_conf.md))). Scenarios often don't need many arguments, but they can be configured in different ways. And at the very least, most need an `objective_target` (the thing you're running a scan against) which you can configure by using the `--target` flag if your initializer registers targets (e.g. `target` initializer)
3. Scenario Techniques (optional). These are supplied by the `--techniques` flag and tell the scenario what to test, but they are always optional. Also note you can obtain these by running `--list-scenarios`

Basic usage will look something like:

```shell
pyrit_scan <scenario> --target <target_name> --initializers <initializer1> <initializer2> --techniques <technique1> <technique2>
```

You can also override scenario parameters directly from the CLI:

```shell
pyrit_scan <scenario> --max-concurrency 10 --max-retries 3 --memory-labels '{"experiment": "test1", "version": "v2"}'
```

Or concretely:

```shell
!pyrit_scan foundry.red_team_agent --target openai_chat --initializers target --techniques base64
```

Example with a basic configuration that runs the Foundry scenario against the objective target defined in the `target` initializer.

In [ ]:
!pyrit_scan foundry.red_team_agent --target openai_chat --initializers target --techniques base64


Running scenario: foundry.red_team_agent

  techniques: 0/1 | attacks: 0 | success rate: 0% | IN_PROGRESS
  techniques: 0/1 | attacks: 0 | success rate: 0% | IN_PROGRESS
  techniques: 0/1 | attacks: 0 | success rate: 0% | IN_PROGRESS
  techniques: 0/1 | attacks: 0 | success rate: 0% | IN_PROGRESS
  techniques: 0/1 | attacks: 0 | success rate: 0% | IN_PROGRESS
  techniques: 0/1 | attacks: 0 | success rate: 0% | IN_PROGRESS
  techniques: 0/1 | attacks: 0 | success rate: 0% | IN_PROGRESS
  techniques: 0/1 | attacks: 0 | success rate: 0% | IN_PROGRESS
  techniques: 0/1 | attacks: 0 | success rate: 0% | IN_PROGRESS
  techniques: 0/1 | attacks: 0 | success rate: 0% | IN_PROGRESS
  techniques: 0/1 | attacks: 0 | success rate: 0% | IN_PROGRESS
  techniques: 0/1 | attacks: 0 | success rate: 0% | IN_PROGRESS
Error (UnicodeEncodeError): 'charmap' codec can't encode characters in position 22-51: character maps to <undefined>


Or with all options and multiple techniques:

```shell
pyrit_scan foundry.red_team_agent --target openai_chat --initializers target --techniques easy crescendo
```

You can also override scenario execution parameters:

```shell
# Override concurrency and retry settings
pyrit_scan foundry.red_team_agent --target openai_chat --initializers target --max-concurrency 10 --max-retries 3

# Add custom memory labels for tracking (must be valid JSON)
pyrit_scan foundry.red_team_agent --target openai_chat --initializers target --memory-labels '{"experiment": "test1", "version": "v2", "researcher": "alice"}'
```

Available CLI parameter overrides:
- `--max-concurrency <int>`: Maximum number of concurrent attack executions
- `--max-retries <int>`: Maximum number of automatic retries if the scenario raises an exception
- `--memory-labels <json>`: Additional labels to apply to all attack runs (must be a JSON string with string keys and values)

You can also use custom initialization scripts by passing file paths. It is relative to your current working directory, but to avoid confusion, full paths are always better:

```shell
pyrit_scan garak.encoding --initialization-scripts ./my_custom_config.py
```

#### Attaching Converters to a Technique

Techniques (techniques) can have a registered converter instance appended to them with the
`<technique>:converter.<name>` syntax. The converter is added to the request side of every attack
the technique produces, on top of any converters the technique already bakes in. This also works on
aggregate techniques (the converter is applied to every technique the aggregate expands to).

First discover the registered converter instances with `--list-converters` (converters are
registered by initializers, so pass the same `--initializers`/`--initialization-scripts` you use to run):

```shell
pyrit_scan --list-converters --initializers my_converters
```

Then reference a converter by name in `--techniques`:

```shell
# Add the registered "translation_spanish" converter to role_play_movie_script only
pyrit_scan airt.rapid_response --target openai_chat --initializers load_default_datasets target my_converters --techniques role_play_movie_script:converter.translation_spanish

# Chain multiple converters (applied in order) and combine with plain techniques
pyrit_scan airt.rapid_response --target openai_chat --initializers load_default_datasets target my_converters --techniques role_play_movie_script:converter.translation_spanish:converter.base64 many_shot
```

#### Using Custom Scenarios

You can define your own scenarios in initialization scripts. The CLI will automatically discover any `Scenario` subclasses and make them available:


In [ ]:
# my_custom_scenarios.py

from pyrit.common import apply_defaults
from pyrit.prompt_target.openai.openai_chat_target import OpenAIChatTarget
from pyrit.scenario import DatasetAttackConfiguration, Scenario, ScenarioTechnique
from pyrit.score import SelfAskRefusalScorer, TrueFalseInverterScorer
from pyrit.setup import initialize_pyrit_async


class MyCustomTechnique(ScenarioTechnique):
    """Techniques for my custom scenario."""

    ALL = ("all", {"all"})
    Technique1 = ("technique1", set[str]())
    Technique2 = ("technique2", set[str]())


class MyCustomScenario(Scenario):
    """My custom scenario that does XYZ."""

    @apply_defaults
    def __init__(self, *, scenario_result_id=None, **kwargs):
        # Scenario-specific configuration only - no runtime parameters
        super().__init__(
            name="My Custom Scenario",
            version=1,
            objective_scorer=TrueFalseInverterScorer(scorer=SelfAskRefusalScorer(chat_target=OpenAIChatTarget())),
            technique_class=MyCustomTechnique,
            default_dataset_config=DatasetAttackConfiguration(dataset_names=["harmbench"]),
            scenario_result_id=scenario_result_id,
        )
        # ... your scenario-specific initialization code

    async def _build_atomic_attacks_async(self, *, context):
        # The single abstract extension point every scenario implements.
        # Read runtime inputs from `context`; return the list of AtomicAttack to run.
        # Matrix-shaped scenarios can delegate to build_matrix_atomic_attacks(context=...).
        # Example: create attacks for each technique composite
        return []


await initialize_pyrit_async(memory_db_type="InMemory")  # type: ignore
MyCustomScenario()

Found default environment files: ['./.pyrit/.env', './.pyrit/.env.local']
Loaded environment file: ./.pyrit/.env
Loaded environment file: ./.pyrit/.env.local


[pyrit:alembic] No new upgrade operations detected.


Then discover and run it:

```shell
# List to see it's available
pyrit_scan --list-scenarios --initialization-scripts ./my_custom_scenarios.py

# Run it with parameter overrides
pyrit_scan my_custom_scenario --initialization-scripts ./my_custom_scenarios.py --max-concurrency 10
```

The scenario name is automatically converted from the class name (e.g., `MyCustomScenario` becomes `my_custom_scenario`).

## Stopping the Backend Server

When you're done, stop the local backend that we started at the top of the notebook.

In [ ]:
!pyrit_scan --stop-server

Server on port 8000 stopped.
